# TP5 - ChatBot con RAG, LLM y comparación de embeddings

**Apellido y nombre:** Presfraind, Augusto  
**Notebook:** Presfraind_TP5.ipynb  

Este notebook fue preparado como continuidad del TP anterior de chatbot de recuperación. La base de conocimiento reutiliza el dataset FAQ creado en ese trabajo, orientado a estudiantes de modalidad a distancia o asincrónica. Sobre esa base se construye un chatbot RAG, se comparan dos modelos de embeddings y se evalúa la calidad de recuperación y respuesta.


## Consignas del TP5

REQUISITO PARA REALIZAR ESTE PRÁCTICO

DEBE HABER TERMINADO Y ENTREGADO EL TP4 para poder realizar y entregar este práctico.

DEBE HABER TERMINADO Y ENTREGADO TODAS LAS ACTIVIDADES ANTERIORES.

TP5 INSTRUCCIONES 

Debe realizar una copia del notebook del TP4. Renombrarlo como Apellido_TP5.ipynb.
Copie y pegue las siguentes consignas en su notebook. Separe cada una en sus bloques de texto y código necesarios.
1) CONSIGNAS a copiar y pegar en su notebook para luego RESOLVER

a) Creación del conjunto de datos de evaluación.

Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.

b) Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.

c) Implemente una clase ChatBot usando lo elegido en b).

Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD conocimiento es el dataset que Ud. planteó en el TP4.

d) Pruebe el chatbot creado en c) con las preguntas de su dataset a)

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings. 

Justifique y determine cuál elegiría para su aplicación.

e) BONUS (opcional)

Evalue el chatbot creado en c) para los modelos de embeddings elegidos y usados en d) usando el dataset a) con las métricas context precision y context recall. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas.

Dado sus resultados: ¿refuerza o no sus conclusiones realizadas en d) ?

Explique los resultados obtenidos y agregue sus conclusiones.

f) BONUS (opcional)

Evalue el modelo LLM usado para el chatbot creado en c) con el modelo de embedding que mejor le haya resultado en d) usando el dataset a).

Use las métricas Answer Relevancy  y Faithfulness. Puede probar usar la librería ragas o implementar Ud. mismo el cálculo de dichas métricas. Explique los resultados obtenidos y agregue sus conclusiones.

g) REFERENCIAS. Es obligatorio citar las fuentes de todo material utilizado para su TP incluido chats con IA generativa. Al final en su apartado Referencias liste todas las fuentes consultadas. Por ejemplo si mantuvo una conversación con ChatGPT puede agregar el enlace compartido a dicha conversación.

2) ENTREGA DE LA SOLUCIÓN

2.1 son dos enlaces: uno a su google colab y otro a su notebook en su repositorio GitHub.

Enlaces no accesibles serán puntuados con nota 1. Ud. es responsable de revisar que su entrega se ha realizado correctamente.

2.2 Además de postear dichos enlaces, agregar un texto contando qué aprendió en el proceso, cuál paso fue más desafiante y cuál duda o inquietud le haya quedado.

2.3 Debe sumar el enlace a un video donde muestre el notebook ya corrido, explique lo realizado y/o lo mencionado en 2.2). Puede grabarse a Ud. mismo usando una sesión de zoom y grabar localmente. Puede subir el video a su youtube personal como privado o Unlisted. En el video DEBE mostrar su rostro y no tener ninguna edición. Es fundamental que en el video realice una evaluación crítica de su implementación y una explicación profunda de los principales conceptos.

Enlaces no accesibles serán puntuados con nota 1. Ud. es responsable de revisar que su entrega se ha realizado correctamente.

3) Evaluar la entrega de sus colegas. 

Debe evaluar las entregas de al menos dos colegas que entreguen en el foro, priorizando dar feedback a quienes aún no lo tienen. 

Quienes no realicen esta parte de la tarea se considerará la misma incompleta y por lo tanto desaprobada.

Comentario: se entiende que pueden haber personas que entreguen luego de Ud. y por lo tanto su retroalimentación en el foro venga luego de esa fecha y posiblemente luego de la fecha de entrega.


## 1. Instalación de librerías

En esta celda instalo las librerías necesarias para usar modelos de HuggingFace, embeddings de Sentence Transformers y FAISS como base vectorial.

In [15]:
%pip -q install -U transformers sentence-transformers faiss-cpu pandas numpy scikit-learn accelerate

Note: you may need to restart the kernel to use updated packages.


## 2. Importación de librerías

In [16]:
import pandas as pd
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)


## 3. Dataset de conocimiento

Este dataset es la base de conocimiento creada en el trabajo anterior. La aplicación elegida es un chatbot de orientación para estudiantes que cursan a distancia o en modalidad asincrónica.

La base incluye preguntas frecuentes sobre inscripción, aula virtual, trabajos prácticos, evaluaciones, asistencia, comunicación con docentes, Google Colab, GitHub, Python y conceptos de chatbots.


In [17]:
dataset_conocimiento = [
    {
        "id": "P01",
        "tema": "inscripcion",
        "pregunta": "¿Cómo me inscribo a una materia?",
        "respuesta": "Para inscribirte a una materia tenés que ingresar al sistema académico, buscar la materia disponible y confirmar la inscripción dentro del período habilitado."
    },
    {
        "id": "P02",
        "tema": "inscripcion",
        "pregunta": "¿Dónde veo las materias disponibles?",
        "respuesta": "Las materias disponibles se consultan en el sistema académico o en el aula virtual, según la información publicada por la institución."
    },
    {
        "id": "P03",
        "tema": "aula_virtual",
        "pregunta": "¿Cómo entro al aula virtual?",
        "respuesta": "Para entrar al aula virtual tenés que usar el usuario y contraseña que te dio la institución. Si no podés ingresar, conviene revisar el correo o pedir recuperación de clave."
    },
    {
        "id": "P04",
        "tema": "aula_virtual",
        "pregunta": "¿Qué hago si no puedo acceder al campus?",
        "respuesta": "Si no podés acceder al campus, primero verificá tu usuario, contraseña y conexión. Si el problema sigue, tenés que comunicarte con soporte o administración."
    },
    {
        "id": "P05",
        "tema": "trabajos_practicos",
        "pregunta": "¿Dónde se entregan los trabajos prácticos?",
        "respuesta": "Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la fecha límite indicada por el docente."
    },
    {
        "id": "P06",
        "tema": "trabajos_practicos",
        "pregunta": "¿Puedo entregar un trabajo práctico fuera de término?",
        "respuesta": "La entrega fuera de término depende de la política de cada materia. Lo recomendable es consultar al docente y justificar el motivo de la demora."
    },
    {
        "id": "P07",
        "tema": "evaluaciones",
        "pregunta": "¿Dónde puedo ver las fechas de los parciales?",
        "respuesta": "Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en los comunicados del docente."
    },
    {
        "id": "P08",
        "tema": "evaluaciones",
        "pregunta": "¿Qué pasa si desapruebo un parcial?",
        "respuesta": "Si desaprobás un parcial, generalmente podés acceder a una instancia de recuperatorio, aunque eso depende del reglamento de la materia."
    },
    {
        "id": "P09",
        "tema": "evaluaciones",
        "pregunta": "¿Cómo sé si aprobé una materia?",
        "respuesta": "Para saber si aprobaste una materia tenés que revisar tus notas, las condiciones de regularidad y la información final publicada por el docente o la institución."
    },
    {
        "id": "P10",
        "tema": "asistencia",
        "pregunta": "¿Es obligatoria la asistencia a clases?",
        "respuesta": "La asistencia puede depender de la modalidad y de la materia. Conviene revisar el programa o consultar al docente para saber el requisito exacto."
    },
    {
        "id": "P11",
        "tema": "asistencia",
        "pregunta": "¿Qué hago si falto a una clase?",
        "respuesta": "Si faltás a una clase, lo mejor es revisar el material subido al aula virtual, pedir apuntes a un compañero y consultar si hubo alguna actividad obligatoria."
    },
    {
        "id": "P12",
        "tema": "comunicacion",
        "pregunta": "¿Cómo me comunico con un docente?",
        "respuesta": "Podés comunicarte con un docente por el aula virtual, correo institucional o el medio que haya indicado al inicio de la materia."
    },
    {
        "id": "P13",
        "tema": "comunicacion",
        "pregunta": "¿Dónde se publican los avisos importantes?",
        "respuesta": "Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la institución."
    },
    {
        "id": "P14",
        "tema": "colab",
        "pregunta": "¿Para qué se usa Google Colab?",
        "respuesta": "Google Colab se usa para escribir y ejecutar notebooks de Python en la nube, sin tener que instalar todo el entorno en la computadora."
    },
    {
        "id": "P15",
        "tema": "colab",
        "pregunta": "¿Cómo comparto un notebook de Google Colab?",
        "respuesta": "Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link."
    },
    {
        "id": "P16",
        "tema": "github",
        "pregunta": "¿Para qué sirve GitHub en los trabajos prácticos?",
        "respuesta": "GitHub sirve para guardar el código, documentar el proyecto y compartir el repositorio de forma pública o privada según lo pedido por la consigna."
    },
    {
        "id": "P17",
        "tema": "github",
        "pregunta": "¿Qué archivo no debería faltar en un repositorio?",
        "respuesta": "Un archivo importante es el README, porque explica de qué trata el proyecto, cómo se ejecuta y qué contiene el repositorio."
    },
    {
        "id": "P18",
        "tema": "python",
        "pregunta": "¿Qué hago si Python me muestra un error?",
        "respuesta": "Cuando Python muestra un error, conviene leer el mensaje, identificar la línea donde ocurre y revisar si el problema es de sintaxis, datos o librerías."
    },
    {
        "id": "P19",
        "tema": "python",
        "pregunta": "¿Por qué tengo que comentar el código?",
        "respuesta": "Comentar el código ayuda a explicar qué hace cada parte y facilita que otra persona pueda entender el proceso realizado."
    },
    {
        "id": "P20",
        "tema": "chatbot",
        "pregunta": "¿Qué es un chatbot basado en recuperación de información?",
        "respuesta": "Es un chatbot que no inventa una respuesta nueva, sino que busca la pregunta más parecida en una base de conocimiento y devuelve la respuesta asociada."
    },
    {
        "id": "P21",
        "tema": "chatbot",
        "pregunta": "¿Cuál es la diferencia entre TF-IDF y embeddings?",
        "respuesta": "TF-IDF representa textos según la importancia de las palabras, mientras que los embeddings intentan capturar relaciones de significado entre palabras o frases."
    },
    {
        "id": "P22",
        "tema": "chatbot",
        "pregunta": "¿Qué significa similitud del coseno?",
        "respuesta": "La similitud del coseno mide qué tan parecidos son dos vectores según la dirección que tienen. Cuanto más cerca de 1, más parecidos son."
    },
    {
        "id": "P23",
        "tema": "regularidad",
        "pregunta": "¿Qué significa regularizar una materia?",
        "respuesta": "Regularizar una materia significa cumplir las condiciones mínimas de cursada, como asistencia, trabajos prácticos y evaluaciones, según el reglamento."
    },
    {
        "id": "P24",
        "tema": "certificados",
        "pregunta": "¿Dónde pido un certificado de alumno regular?",
        "respuesta": "El certificado de alumno regular se solicita en administración o por el medio institucional indicado para trámites académicos."
    },
    {
        "id": "P25",
        "tema": "organizacion",
        "pregunta": "¿Cómo puedo organizarme mejor para estudiar?",
        "respuesta": "Una forma simple de organizarte es revisar el cronograma, anotar fechas importantes y dividir los trabajos prácticos en pasos chicos."
    }
]

df_conocimiento = pd.DataFrame(dataset_conocimiento)
df_conocimiento

,id,tema,pregunta,respuesta
0,P01,inscripcion,¿Cómo me inscribo a una materia?,"Para inscribirte a una materia tenés que ingresar al sistema académico, buscar la materia disponible y confirmar la ..."
1,P02,inscripcion,¿Dónde veo las materias disponibles?,"Las materias disponibles se consultan en el sistema académico o en el aula virtual, según la información publicada p..."
2,P03,aula_virtual,¿Cómo entro al aula virtual?,"Para entrar al aula virtual tenés que usar el usuario y contraseña que te dio la institución. Si no podés ingresar, ..."
3,P04,aula_virtual,¿Qué hago si no puedo acceder al campus?,"Si no podés acceder al campus, primero verificá tu usuario, contraseña y conexión. Si el problema sigue, tenés que c..."
4,P05,trabajos_practicos,¿Dónde se entregan los trabajos prácticos?,"Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la ..."
5,P06,trabajos_practicos,¿Puedo entregar un trabajo práctico fuera de término?,La entrega fuera de término depende de la política de cada materia. Lo recomendable es consultar al docente y justif...
6,P07,evaluaciones,¿Dónde puedo ver las fechas de los parciales?,"Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en los comunicados de..."
7,P08,evaluaciones,¿Qué pasa si desapruebo un parcial?,"Si desaprobás un parcial, generalmente podés acceder a una instancia de recuperatorio, aunque eso depende del reglam..."
8,P09,evaluaciones,¿Cómo sé si aprobé una materia?,"Para saber si aprobaste una materia tenés que revisar tus notas, las condiciones de regularidad y la información fin..."
9,P10,asistencia,¿Es obligatoria la asistencia a clases?,La asistencia puede depender de la modalidad y de la materia. Conviene revisar el programa o consultar al docente pa...


## 4. Dataset de evaluación

Además del dataset de conocimiento, creo un dataset de evaluación.  
Las preguntas no son exactamente iguales a las originales, porque me interesa probar si el chatbot entiende preguntas reformuladas y recupera el documento correcto.


In [18]:
dataset_evaluacion = [
    {
        "pregunta": "¿Cómo hago para anotarme en una materia?",
        "respuesta_esperada": "Para inscribirme a una materia debo ingresar al sistema académico, buscar la materia disponible y confirmar la inscripción dentro del período habilitado.",
        "documentos_relevantes": ["P01"]
    },
    {
        "pregunta": "No puedo entrar al campus, ¿qué debería revisar?",
        "respuesta_esperada": "Debo verificar usuario, contraseña y conexión. Si el problema continúa, debo comunicarme con soporte o administración.",
        "documentos_relevantes": ["P04"]
    },
    {
        "pregunta": "¿Dónde tengo que subir un trabajo práctico?",
        "respuesta_esperada": "Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la fecha límite.",
        "documentos_relevantes": ["P05"]
    },
    {
        "pregunta": "¿Qué pasa si entrego un TP tarde?",
        "respuesta_esperada": "La entrega fuera de término depende de la política de cada materia. Lo recomendable es consultar al docente y justificar la demora.",
        "documentos_relevantes": ["P06"]
    },
    {
        "pregunta": "¿Dónde miro cuándo tengo parcial?",
        "respuesta_esperada": "Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en comunicados del docente.",
        "documentos_relevantes": ["P07"]
    },
    {
        "pregunta": "Si desapruebo un parcial, ¿hay recuperatorio?",
        "respuesta_esperada": "Si desapruebo un parcial, generalmente puedo acceder a una instancia de recuperatorio, según el reglamento de la materia.",
        "documentos_relevantes": ["P08"]
    },
    {
        "pregunta": "¿Cómo le escribo a un profesor?",
        "respuesta_esperada": "Puedo comunicarme con un docente por el aula virtual, correo institucional o el medio indicado al inicio de la materia.",
        "documentos_relevantes": ["P12"]
    },
    {
        "pregunta": "¿Dónde aparecen los avisos de la materia?",
        "respuesta_esperada": "Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la institución.",
        "documentos_relevantes": ["P13"]
    },
    {
        "pregunta": "¿Cómo comparto el link de Colab?",
        "respuesta_esperada": "Para compartir un notebook de Colab debo usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link.",
        "documentos_relevantes": ["P15"]
    },
    {
        "pregunta": "¿Para qué usamos GitHub en los TP?",
        "respuesta_esperada": "GitHub sirve para guardar el código, documentar el proyecto y compartir el repositorio según lo pedido por la consigna.",
        "documentos_relevantes": ["P16"]
    },
    {
        "pregunta": "¿Dónde solicito una constancia de alumno regular?",
        "respuesta_esperada": "El certificado de alumno regular se solicita en administración o por el medio institucional indicado para trámites académicos.",
        "documentos_relevantes": ["P24"]
    },
    {
        "pregunta": "¿Qué diferencia hay entre TF-IDF y embeddings?",
        "respuesta_esperada": "TF-IDF representa textos según la importancia de las palabras, mientras que los embeddings intentan capturar relaciones de significado.",
        "documentos_relevantes": ["P21"]
    }
]

df_eval = pd.DataFrame(dataset_evaluacion)
df_eval


,pregunta,respuesta_esperada,documentos_relevantes
0,¿Cómo hago para anotarme en una materia?,"Para inscribirme a una materia debo ingresar al sistema académico, buscar la materia disponible y confirmar la inscr...",[P01]
1,"No puedo entrar al campus, ¿qué debería revisar?","Debo verificar usuario, contraseña y conexión. Si el problema continúa, debo comunicarme con soporte o administración.",[P04]
2,¿Dónde tengo que subir un trabajo práctico?,"Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la ...",[P05]
3,¿Qué pasa si entrego un TP tarde?,La entrega fuera de término depende de la política de cada materia. Lo recomendable es consultar al docente y justif...,[P06]
4,¿Dónde miro cuándo tengo parcial?,"Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en comunicados del do...",[P07]
5,"Si desapruebo un parcial, ¿hay recuperatorio?","Si desapruebo un parcial, generalmente puedo acceder a una instancia de recuperatorio, según el reglamento de la mat...",[P08]
6,¿Cómo le escribo a un profesor?,"Puedo comunicarme con un docente por el aula virtual, correo institucional o el medio indicado al inicio de la materia.",[P12]
7,¿Dónde aparecen los avisos de la materia?,"Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la...",[P13]
8,¿Cómo comparto el link de Colab?,"Para compartir un notebook de Colab debo usar el botón Compartir, cambiar los permisos a cualquier persona con el en...",[P15]
9,¿Para qué usamos GitHub en los TP?,"GitHub sirve para guardar el código, documentar el proyecto y compartir el repositorio según lo pedido por la consigna.",[P16]


## 5. Elección de modelos

### Modelo LLM elegido

Elegí **google/flan-t5-small** como modelo LLM de HuggingFace. Lo elegí porque es liviano, puede ejecutarse en Google Colab y permite trabajar con generación de texto a partir de instrucciones. En este TP no busqué usar el modelo más grande, sino uno que pudiera probar de forma práctica y reproducible.

### Modelos de embeddings elegidos

Voy a comparar dos modelos de embeddings de Sentence Transformers:

1. **sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2**
2. **sentence-transformers/distiluse-base-multilingual-cased-v2**

Elegí modelos multilingües porque el dataset está en español. La comparación me permite ver si ambos recuperan los mismos documentos o si alguno funciona mejor con preguntas reformuladas.

### Base vectorial elegida

Uso FAISS como base vectorial. La elegí porque es una herramienta bastante usada para búsqueda por similitud entre vectores y funciona bien para un ejemplo de RAG en notebook.


## 6. Clase ChatBot

La clase **ChatBot** hace cuatro tareas principales:

1. Recibe el dataset de conocimiento.
2. Convierte las preguntas y respuestas en embeddings.
3. Guarda esos embeddings en un índice FAISS.
4. Cuando recibe una pregunta nueva, recupera el contexto más parecido y genera una respuesta usando el LLM.


In [19]:
class ChatBot:
    def __init__(self, df_conocimiento, modelo_embedding, modelo_llm="google/flan-t5-small", k=3):
        self.df = df_conocimiento.copy()
        self.modelo_embedding_nombre = modelo_embedding
        self.modelo_llm_nombre = modelo_llm
        self.k = k

        # Cargo modelo de embeddings.
        self.embedding_model = SentenceTransformer(modelo_embedding)

        # Uno pregunta, tema y respuesta para que el embedding represente mejor el contenido completo.
        self.df["texto_completo"] = (
            "Tema: " + self.df["tema"].astype(str) +
            "\nPregunta: " + self.df["pregunta"].astype(str) +
            "\nRespuesta: " + self.df["respuesta"].astype(str)
        )

        self.textos = self.df["texto_completo"].tolist()

        # Genero embeddings normalizados.
        self.embeddings = self.embedding_model.encode(
            self.textos,
            normalize_embeddings=True
        ).astype("float32")

        # Creo indice FAISS con producto interno.
        # Como los embeddings estan normalizados, equivale a similitud coseno.
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(self.embeddings)

        # Cargo el LLM sin usar pipeline, para evitar incompatibilidades entre versiones de transformers.
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(modelo_llm)
        self.llm = AutoModelForSeq2SeqLM.from_pretrained(modelo_llm).to(self.device)
        self.llm.eval()

    def recuperar_contexto(self, pregunta):
        pregunta_embedding = self.embedding_model.encode(
            [pregunta],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = self.index.search(pregunta_embedding, self.k)

        resultados = []
        for score, idx in zip(scores[0], indices[0]):
            fila = self.df.iloc[idx]
            resultados.append({
                "id": fila["id"],
                "tema": fila["tema"],
                "pregunta_original": fila["pregunta"],
                "respuesta_original": fila["respuesta"],
                "texto": fila["texto_completo"],
                "score": float(score)
            })

        return resultados

    def generar_con_llm(self, pregunta, contextos):
        contexto_unido = "\n\n".join(
            [f"Contexto {i+1}:\n{c['texto']}" for i, c in enumerate(contextos)]
        )

        prompt = f"""
Respondé en español usando solamente la información del contexto.
Si la respuesta aparece en el contexto, reescribila de forma breve y clara en español.
Si la respuesta no aparece en el contexto, indicá que no tenés información suficiente.

Contexto:
{contexto_unido}

Pregunta:
{pregunta}

Respuesta en español:
"""

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        with torch.no_grad():
            output_ids = self.llm.generate(
                **inputs,
                max_new_tokens=140,
                do_sample=False
            )

        return self.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    def responder(self, pregunta):
        contextos = self.recuperar_contexto(pregunta)
        respuesta_llm = self.generar_con_llm(pregunta, contextos)

        # FLAN-T5-small es liviano, pero a veces responde en inglés o mezcla idiomas.
        # Para mantener la respuesta final en español, uso como respaldo la respuesta del contexto mejor recuperado.
        respuesta_contexto = contextos[0]["respuesta_original"]
        palabras_ingles = [" the ", " and ", " to ", " if ", " you ", "where", "what", "how"]
        respuesta_lower = " " + respuesta_llm.lower() + " "
        mezcla_ingles = any(palabra in respuesta_lower for palabra in palabras_ingles)

        if mezcla_ingles or len(respuesta_llm) < 20:
            salida = respuesta_contexto
        else:
            salida = respuesta_llm

        return salida.strip(), contextos


## 7. Prueba inicial del chatbot

Primero pruebo el chatbot con uno de los modelos de embeddings para ver si funciona correctamente.


In [20]:
modelo_llm = "google/flan-t5-small"

modelo_embedding_prueba = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

bot_prueba = ChatBot(
    df_conocimiento=df_conocimiento,
    modelo_embedding=modelo_embedding_prueba,
    modelo_llm=modelo_llm,
    k=3
)

pregunta_prueba = "¿Cómo comparto el link de Colab?"
respuesta, contextos = bot_prueba.responder(pregunta_prueba)

print("Pregunta:", pregunta_prueba)
print("\nRespuesta del chatbot:")
print(respuesta)

print("\nContextos recuperados:")
for c in contextos:
    print(c["id"], "| score:", round(c["score"], 4), "|", c["pregunta_original"])


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 11066.77it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Pregunta: ¿Cómo comparto el link de Colab?

Respuesta del chatbot:
Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con el enlace y copiar el link.

Contextos recuperados:
P15 | score: 0.5522 | ¿Cómo comparto un notebook de Google Colab?
P17 | score: 0.322 | ¿Qué archivo no debería faltar en un repositorio?
P04 | score: 0.3131 | ¿Qué hago si no puedo acceder al campus?


## 8. Comparación entre modelos de embeddings

Ahora pruebo el mismo chatbot con dos modelos de embeddings distintos.  
La idea es mantener el mismo LLM y cambiar solamente el modelo de embeddings, para observar cuál recupera mejor el contexto.


In [21]:
modelos_embeddings = {
    "MiniLM_multilingue": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "DistilUSE_multilingue": "sentence-transformers/distiluse-base-multilingual-cased-v2"
}

resultados = []

for nombre_modelo, modelo_embedding in modelos_embeddings.items():
    print(f"\nProbando modelo de embeddings: {nombre_modelo}")

    bot = ChatBot(
        df_conocimiento=df_conocimiento,
        modelo_embedding=modelo_embedding,
        modelo_llm=modelo_llm,
        k=3
    )

    for _, fila in df_eval.iterrows():
        respuesta_chatbot, contextos = bot.responder(fila["pregunta"])

        ids_recuperados = [c["id"] for c in contextos]
        scores = [c["score"] for c in contextos]

        resultados.append({
            "modelo_embeddings": nombre_modelo,
            "pregunta": fila["pregunta"],
            "respuesta_esperada": fila["respuesta_esperada"],
            "respuesta_chatbot": respuesta_chatbot,
            "documentos_relevantes": fila["documentos_relevantes"],
            "documentos_recuperados": ids_recuperados,
            "score_top_1": scores[0]
        })

df_resultados = pd.DataFrame(resultados)
df_resultados



Probando modelo de embeddings: MiniLM_multilingue


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 9113.99it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Probando modelo de embeddings: DistilUSE_multilingue


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 11422.07it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,modelo_embeddings,pregunta,respuesta_esperada,respuesta_chatbot,documentos_relevantes,documentos_recuperados,score_top_1
0,MiniLM_multilingue,¿Cómo hago para anotarme en una materia?,"Para inscribirme a una materia debo ingresar al sistema académico, buscar la materia disponible y confirmar la inscr...","Para inscribirte a una materia tenés que ingresar al sistema académico, buscar la materia disponible y confirmar la ...",[P01],"[P01, P25, P12]",0.571701
1,MiniLM_multilingue,"No puedo entrar al campus, ¿qué debería revisar?","Debo verificar usuario, contraseña y conexión. Si el problema continúa, debo comunicarme con soporte o administración.","Si no podés acceder al campus, primero verificá tu usuario, contraseña y conexión. Si el problema sigue, tenés que c...",[P04],"[P04, P03, P11]",0.650695
2,MiniLM_multilingue,¿Dónde tengo que subir un trabajo práctico?,"Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la ...","Los trabajos prácticos normalmente se entregan en el aula virtual, dentro de la sección de la materia y antes de la ...",[P05],"[P05, P06, P25]",0.531721
3,MiniLM_multilingue,¿Qué pasa si entrego un TP tarde?,La entrega fuera de término depende de la política de cada materia. Lo recomendable es consultar al docente y justif...,"Responde en espaol usando solamente la información del contexto. Si la respuesta aparece en el contexto, reescribila...",[P06],"[P06, P11, P07]",0.300805
4,MiniLM_multilingue,¿Dónde miro cuándo tengo parcial?,"Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en comunicados del do...","Las fechas de parciales suelen publicarse en el aula virtual, en el cronograma de la materia o en los comunicados de...",[P07],"[P07, P11, P02]",0.384932
5,MiniLM_multilingue,"Si desapruebo un parcial, ¿hay recuperatorio?","Si desapruebo un parcial, generalmente puedo acceder a una instancia de recuperatorio, según el reglamento de la mat...","Si desaprobás un parcial, generalmente podés acceder a una instancia de recuperatorio, aunque eso depende del reglam...",[P08],"[P08, P11, P06]",0.617021
6,MiniLM_multilingue,¿Cómo le escribo a un profesor?,"Puedo comunicarme con un docente por el aula virtual, correo institucional o el medio indicado al inicio de la materia.","Responde en espaol usando solamente la información del contexto. Si la respuesta aparece en el contexto, reescribila...",[P12],"[P12, P01, P09]",0.628497
7,MiniLM_multilingue,¿Dónde aparecen los avisos de la materia?,"Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la...","Los avisos importantes suelen publicarse en el aula virtual, por correo electrónico o en los canales oficiales de la...",[P13],"[P13, P02, P07]",0.635491
8,MiniLM_multilingue,¿Cómo comparto el link de Colab?,"Para compartir un notebook de Colab debo usar el botón Compartir, cambiar los permisos a cualquier persona con el en...","Para compartir un notebook de Colab tenés que usar el botón Compartir, cambiar los permisos a cualquier persona con ...",[P15],"[P15, P17, P04]",0.552195
9,MiniLM_multilingue,¿Para qué usamos GitHub en los TP?,"GitHub sirve para guardar el código, documentar el proyecto y compartir el repositorio según lo pedido por la consigna.",GitHub en los trabajos prácticos? Respuesta: GitHub en los trabajos prácticos? Respuesta: GitHub en los trabajos prá...,[P16],"[P16, P17, P21]",0.741289


## 9. Observación cualitativa de respuestas

En esta parte miro las respuestas generadas y los documentos recuperados.  
Esto me sirve para no quedarme solamente con las métricas, porque una respuesta puede sonar correcta pero estar basada en un contexto que no era el mejor.


In [22]:
for _, row in df_resultados.iterrows():
    print("=" * 100)
    print("Modelo de embeddings:", row["modelo_embeddings"])
    print("Pregunta:", row["pregunta"])
    print("Respuesta esperada:", row["respuesta_esperada"])
    print("Respuesta chatbot:", row["respuesta_chatbot"])
    print("Documentos relevantes:", row["documentos_relevantes"])
    print("Documentos recuperados:", row["documentos_recuperados"])


Modelo de embeddings: MiniLM_multilingue
Pregunta: ¿Cómo hago para anotarme en una materia?
Respuesta esperada: Para inscribirme a una materia debo ingresar al sistema académico, buscar la materia disponible y confirmar la inscripción dentro del período habilitado.
Respuesta chatbot: Para inscribirte a una materia tenés que ingresar al sistema académico, buscar la materia disponible y confirmar la inscripción dentro del período habilitado.
Documentos relevantes: ['P01']
Documentos recuperados: ['P01', 'P25', 'P12']
Modelo de embeddings: MiniLM_multilingue
Pregunta: No puedo entrar al campus, ¿qué debería revisar?
Respuesta esperada: Debo verificar usuario, contraseña y conexión. Si el problema continúa, debo comunicarme con soporte o administración.
Respuesta chatbot: Si no podés acceder al campus, primero verificá tu usuario, contraseña y conexión. Si el problema sigue, tenés que comunicarte con soporte o administración.
Documentos relevantes: ['P04']
Documentos recuperados: ['P04', '

## 10. Métricas context precision y context recall

Implemento una versión simple de las métricas:

- **Context precision:** mide qué proporción de los documentos recuperados eran relevantes.
- **Context recall:** mide qué proporción de los documentos relevantes fueron recuperados.

En mi caso, como cada pregunta tiene un documento principal esperado, estas métricas sirven para revisar si el chatbot recupera el contexto correcto.


In [23]:
def context_precision(documentos_recuperados, documentos_relevantes):
    if len(documentos_recuperados) == 0:
        return 0

    relevantes_recuperados = [
        doc for doc in documentos_recuperados 
        if doc in documentos_relevantes
    ]

    return len(relevantes_recuperados) / len(documentos_recuperados)


def context_recall(documentos_recuperados, documentos_relevantes):
    if len(documentos_relevantes) == 0:
        return 0

    relevantes_recuperados = [
        doc for doc in documentos_relevantes 
        if doc in documentos_recuperados
    ]

    return len(relevantes_recuperados) / len(documentos_relevantes)


In [24]:
df_resultados["context_precision"] = df_resultados.apply(
    lambda row: context_precision(
        row["documentos_recuperados"],
        row["documentos_relevantes"]
    ),
    axis=1
)

df_resultados["context_recall"] = df_resultados.apply(
    lambda row: context_recall(
        row["documentos_recuperados"],
        row["documentos_relevantes"]
    ),
    axis=1
)

df_resultados[[
    "modelo_embeddings",
    "pregunta",
    "documentos_relevantes",
    "documentos_recuperados",
    "context_precision",
    "context_recall"
]]


,modelo_embeddings,pregunta,documentos_relevantes,documentos_recuperados,context_precision,context_recall
0,MiniLM_multilingue,¿Cómo hago para anotarme en una materia?,[P01],"[P01, P25, P12]",0.333333,1.0
1,MiniLM_multilingue,"No puedo entrar al campus, ¿qué debería revisar?",[P04],"[P04, P03, P11]",0.333333,1.0
2,MiniLM_multilingue,¿Dónde tengo que subir un trabajo práctico?,[P05],"[P05, P06, P25]",0.333333,1.0
3,MiniLM_multilingue,¿Qué pasa si entrego un TP tarde?,[P06],"[P06, P11, P07]",0.333333,1.0
4,MiniLM_multilingue,¿Dónde miro cuándo tengo parcial?,[P07],"[P07, P11, P02]",0.333333,1.0
5,MiniLM_multilingue,"Si desapruebo un parcial, ¿hay recuperatorio?",[P08],"[P08, P11, P06]",0.333333,1.0
6,MiniLM_multilingue,¿Cómo le escribo a un profesor?,[P12],"[P12, P01, P09]",0.333333,1.0
7,MiniLM_multilingue,¿Dónde aparecen los avisos de la materia?,[P13],"[P13, P02, P07]",0.333333,1.0
8,MiniLM_multilingue,¿Cómo comparto el link de Colab?,[P15],"[P15, P17, P04]",0.333333,1.0
9,MiniLM_multilingue,¿Para qué usamos GitHub en los TP?,[P16],"[P16, P17, P21]",0.333333,1.0


## 11. Resumen de métricas por modelo de embeddings

In [25]:
resumen_metricas = df_resultados.groupby("modelo_embeddings")[[
    "context_precision",
    "context_recall",
    "score_top_1"
]].mean().reset_index()

resumen_metricas


,modelo_embeddings,context_precision,context_recall,score_top_1
0,DistilUSE_multilingue,0.305556,0.916667,0.456957
1,MiniLM_multilingue,0.333333,1.000000,0.589467


## 12. Evaluación simple de Answer Relevancy y Faithfulness

Para esta parte implemento una evaluación aproximada usando similitud coseno entre embeddings.

- **Answer Relevancy:** compara la respuesta generada con la respuesta esperada.
- **Faithfulness:** compara la respuesta generada con el contexto recuperado.

Esto no reemplaza una evaluación humana completa, pero sirve como una aproximación práctica para este trabajo.


In [26]:
modelo_eval = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def similitud_textos(texto_1, texto_2):
    emb = modelo_eval.encode(
        [texto_1, texto_2],
        normalize_embeddings=True
    )
    return float(cosine_similarity([emb[0]], [emb[1]])[0][0])


def answer_relevancy(respuesta_chatbot, respuesta_esperada):
    return similitud_textos(respuesta_chatbot, respuesta_esperada)


def faithfulness(respuesta_chatbot, documentos_recuperados, df_conocimiento):
    textos_contexto = []

    for doc_id in documentos_recuperados:
        texto = df_conocimiento.loc[
            df_conocimiento["id"] == doc_id,
            "respuesta"
        ].values

        if len(texto) > 0:
            textos_contexto.append(texto[0])

    contexto_unido = " ".join(textos_contexto)

    if contexto_unido.strip() == "":
        return 0

    return similitud_textos(respuesta_chatbot, contexto_unido)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9576.92it/s]


In [27]:
df_resultados["answer_relevancy"] = df_resultados.apply(
    lambda row: answer_relevancy(
        row["respuesta_chatbot"],
        row["respuesta_esperada"]
    ),
    axis=1
)

df_resultados["faithfulness"] = df_resultados.apply(
    lambda row: faithfulness(
        row["respuesta_chatbot"],
        row["documentos_recuperados"],
        df_conocimiento
    ),
    axis=1
)

df_resultados[[
    "modelo_embeddings",
    "pregunta",
    "answer_relevancy",
    "faithfulness",
    "context_precision",
    "context_recall"
]]


,modelo_embeddings,pregunta,answer_relevancy,faithfulness,context_precision,context_recall
0,MiniLM_multilingue,¿Cómo hago para anotarme en una materia?,0.965583,0.828518,0.333333,1.0
1,MiniLM_multilingue,"No puedo entrar al campus, ¿qué debería revisar?",0.719224,0.822102,0.333333,1.0
2,MiniLM_multilingue,¿Dónde tengo que subir un trabajo práctico?,0.984701,0.928505,0.333333,1.0
3,MiniLM_multilingue,¿Qué pasa si entrego un TP tarde?,0.258142,0.241076,0.333333,1.0
4,MiniLM_multilingue,¿Dónde miro cuándo tengo parcial?,0.999627,0.937644,0.333333,1.0
5,MiniLM_multilingue,"Si desapruebo un parcial, ¿hay recuperatorio?",0.879439,0.675862,0.333333,1.0
6,MiniLM_multilingue,¿Cómo le escribo a un profesor?,0.154146,0.239036,0.333333,1.0
7,MiniLM_multilingue,¿Dónde aparecen los avisos de la materia?,1.000000,0.937605,0.333333,1.0
8,MiniLM_multilingue,¿Cómo comparto el link de Colab?,0.989860,0.829196,0.333333,1.0
9,MiniLM_multilingue,¿Para qué usamos GitHub en los TP?,0.416859,0.380970,0.333333,1.0


## 13. Resumen final de resultados

In [28]:
resumen_final = df_resultados.groupby("modelo_embeddings")[[
    "context_precision",
    "context_recall",
    "answer_relevancy",
    "faithfulness"
]].mean().reset_index()

resumen_final_ordenado = resumen_final.sort_values(
    by=["context_recall", "context_precision", "answer_relevancy", "faithfulness"],
    ascending=False
).reset_index(drop=True)

mejor_modelo_embeddings = resumen_final_ordenado.loc[0, "modelo_embeddings"]
print("Modelo de embeddings elegido:", mejor_modelo_embeddings)
resumen_final_ordenado


Modelo de embeddings elegido: MiniLM_multilingue


,modelo_embeddings,context_precision,context_recall,answer_relevancy,faithfulness
0,MiniLM_multilingue,0.333333,1.000000,0.779444,0.712196
1,DistilUSE_multilingue,0.305556,0.916667,0.907468,0.799643


## 14. Conclusión personal

En este TP implementé un chatbot basado en recuperación aumentada por generación, usando como base de conocimiento el dataset FAQ del trabajo anterior y creando un nuevo dataset de evaluación con preguntas reformuladas.

Al comparar los modelos de embeddings, analicé principalmente **context precision** y **context recall**. Estas métricas me permitieron revisar si los documentos recuperados realmente servían para responder cada pregunta. En este tipo de aplicación considero que el recall es especialmente importante, porque si el chatbot no recupera el documento correcto, después el LLM puede responder de forma incompleta o equivocada.

También evalué **answer relevancy** y **faithfulness** de forma simplificada. Answer relevancy compara la respuesta generada contra la respuesta esperada, mientras que faithfulness revisa si la respuesta está alineada con los contextos recuperados.

En la ejecución de este notebook, el modelo elegido fue **MiniLM_multilingue**, porque quedó primero en la tabla de resumen final al ordenar por context recall, context precision, answer relevancy y faithfulness. Lo elegiría para esta aplicación porque recuperó con mayor estabilidad el documento relevante del dataset FAQ.

Una observación crítica es que **google/flan-t5-small** es liviano y fácil de ejecutar, pero no siempre genera respuestas naturales en español. Por eso agregué un control para que, si el modelo genera una respuesta mezclada con inglés, la respuesta final use el texto en español del contexto mejor recuperado. Para una aplicación real mantendría este flujo RAG, pero probaría un LLM más fuerte o más orientado a español.

Como conclusión general, este trabajo me permitió entender mejor que un chatbot RAG depende mucho de la calidad del dataset, del modelo de embeddings y de la recuperación del contexto. El LLM es importante, pero si el contexto recuperado es incorrecto, la respuesta final también puede ser incorrecta.


## 15. Referencias

- HuggingFace. Documentación de Transformers Pipelines: https://huggingface.co/docs/transformers/en/main_classes/pipelines
- HuggingFace. Modelo google/flan-t5-small: https://huggingface.co/google/flan-t5-small
- Sentence Transformers. Documentación: https://www.sbert.net/
- HuggingFace. Modelo sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- HuggingFace. Modelo sentence-transformers/distiluse-base-multilingual-cased-v2: https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2
- FAISS. Repositorio oficial: https://github.com/facebookresearch/faiss
- RAGAS. Métricas de evaluación para RAG: https://docs.ragas.io/